# Lesson 10.2: Gather evidence without confusing checks and permission

A coordinator can collect order and policy evidence together because neither read requires the other. It still needs both before it can prepare the draft. Follow the source identifiers in the returned result and then check why permission for an actual handoff requires another step.

[Companion lesson](../modules/10-agent-orchestration/02-code-workshop.md) · [Original-source walkthrough](../modules/10-agent-orchestration/05-source-walkthrough.md)

Run one cell at a time from top to bottom. The programme uses the fictional records shown in the code. No API key or model download is needed.

## Predict the result

Which two sources make up the draft? What happens when the programme is asked to authorise a handoff without an approval?

In [ ]:
import asyncio

async def order_evidence(case):
    await asyncio.sleep(0)
    return {"source": "order-17", "reported_difference": 2}

async def policy_evidence(case):
    await asyncio.sleep(0)
    return {"source": "policy-v3", "requires_review": True}

async def prepare_case(case, fetch_order, fetch_policy):
    results = await asyncio.gather(
        fetch_order(case), fetch_policy(case), return_exceptions=True
    )
    failures = [type(item).__name__ for item in results if isinstance(item, Exception)]
    if failures:
        return {"status": "incomplete", "failures": failures}
    order, policy = results
    return {
        "status": "draft",
        "case_id": case["id"],
        "sources": [order["source"], policy["source"]],
        "reported_difference": order["reported_difference"],
        "requires_review": policy["requires_review"],
    }

def authorise_handoff(draft, approval):
    if draft["status"] != "draft":
        raise ValueError("Evidence collection is incomplete")
    if not approval or approval["case_id"] != draft["case_id"]:
        raise PermissionError("No matching approval")
    if approval["decision"] != "approve":
        raise PermissionError("Review did not approve this handoff")
    return {"status": "ready_for_execution", "case_id": draft["case_id"]}

case = {"id": "C-17"}
draft = await prepare_case(case, order_evidence, policy_evidence)
print(draft["status"], draft["sources"])
try:
    authorise_handoff(draft, None)
except PermissionError as error:
    print(type(error).__name__)

## Inspect the result

`print` displays values below the cell. When present, `assert` checks a specific expectation and raises an error if it fails.

In [ ]:
assert draft["status"] == "draft"
assert draft["sources"] == ["order-17", "policy-v3"]

## Try a change

Try an approval that names a different case. Should it allow execution?

Make a prediction before opening the worked answer.

In [ ]:
# Try the change here, or make a temporary edit in the first code cell.

<details>
<summary>Worked answer</summary>

The draft can be prepared successfully while the action stays blocked. The notebook uses `await` because Jupyter already has a loop that manages waiting tasks.

```python
try:
    authorise_handoff(draft, {"case_id": "another-case", "approved": True})
except PermissionError as error:
    print(error)
else:
    raise AssertionError("Approval for another case should be refused")
```

</details>